# LOAN APPROVAL PROJECT

## 1. Problem Definition (Classification Problem)
> This is a project to determine whether a customer should be able to apply for loan or not

## 2. Data
> This data is from kaggle learning competion
  https://www.kaggle.com/competitions/playground-series-s4e10/data

## 3. Evaluation
> The Evaluation metrics for this project is accuracy
> And the aim is to reach atleast a > 95% aaccuracy

## 4. Features

In [54]:
# import all necessary tools
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# import the models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# import model evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import RocCurveDisplay
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, cross_val_score

# import data cleaning tool
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [55]:
# import the data
df = pd.read_csv("data/train.csv")
df.head()

,id,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
0,0,37,35000,RENT,0.0,EDUCATION,B,6000,11.49,0.17,N,14,0
1,1,22,56000,OWN,6.0,MEDICAL,C,4000,13.35,0.07,N,2,0
2,2,29,28800,OWN,8.0,PERSONAL,A,6000,8.90,0.21,N,10,0
3,3,30,70000,RENT,14.0,VENTURE,B,12000,11.11,0.17,N,5,0
4,4,22,60000,RENT,2.0,MEDICAL,A,6000,6.92,0.10,N,3,0


In [56]:
df.describe

<bound method NDFrame.describe of           id  person_age  person_income person_home_ownership  \
0          0          37          35000                  RENT   
1          1          22          56000                   OWN   
2          2          29          28800                   OWN   
3          3          30          70000                  RENT   
4          4          22          60000                  RENT   
...      ...         ...            ...                   ...   
58640  58640          34         120000              MORTGAGE   
58641  58641          28          28800                  RENT   
58642  58642          23          44000                  RENT   
58643  58643          22          30000                  RENT   
58644  58644          31          75000              MORTGAGE   

       person_emp_length loan_intent loan_grade  loan_amnt  loan_int_rate  \
0                    0.0   EDUCATION          B       6000          11.49   
1                    6.0     ME

In [57]:
df["loan_status"]

0        0
1        0
2        0
3        0
4        0
        ..
58640    0
58641    1
58642    1
58643    0
58644    0
Name: loan_status, Length: 58645, dtype: int64

In [ ]:
df["loan_status"].hist();

In [ ]:
df.isna().sum()
# since everything is 0 that means no null columns -- no need for preprocessing

In [ ]:
# Delete the ID Column
ids = df["id"]
df = df.drop(columns=["id"])

# split into X $ y
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_test.head()

In [ ]:
categoric_col = X_train.select_dtypes(include=["object", "category", "str"]).columns
categoric_col

In [ ]:
# Identify the numeric columns and Category Columns
numeric_col = X_train.select_dtypes(include=["number"]).columns
categoric_col = X_train.select_dtypes(include=["object", "category", "str"]).columns

# Since there is no missing values we don't need imputer to fill columns

# Use OneHotEncoder to convert categorical to numbers
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Use Transformer to map categorical tranformer to categorical columns
preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categoric_col)
])

## Modelling

In [ ]:
# instantiate the models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistics Regression": LogisticRegression(random_state=42),
    "KNeighbors": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(random_state=42),
    "LightGMM": LGBMClassifier(random_state=42)
}

# Create a function to fit and score the model
def fit_and_score(models, X_train, y_train, X_test, y_test):
    """
    This Function returns scores for each of the model.
    """
    model_scores = {}
    
    # fit the model to the data
    for name, model in models.items():
        
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ])
        
        pipeline.fit(X_train, y_train)

        # score the model
        model_scores[name] = pipeline.score(X_test, y_test)
    return model_scores

In [ ]:
model_scores = fit_and_score(models=models, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)
model_scores

## Evaluation